# Manim CPU (Cairo) vs OpenGL (GPU) render benchmark — Colab version

**Before running anything:** `Runtime` menu (top-left) -> `Change runtime type` -> Hardware
accelerator -> **T4 GPU** -> Save. Then `Runtime` -> `Run all`.

This notebook clones the project's own public GitHub repo itself — you don't need to upload
any files, just this notebook. Same test as the Kaggle run (2026-09-04): render 3 real,
previously-shipped scenes at 3 duration tiers, once with the default Cairo renderer and once
with `--renderer=opengl`, and compare wall-clock time. On Kaggle, OpenGL failed outright
(container only exposed CUDA compute, not hardware OpenGL — fell back to software `llvmpipe`).
Colab is commonly used for real headless-GPU OpenGL rendering (robotics/RL sims), so this is
worth an independent try.

In [ ]:
!nvidia-smi || echo "NO GPU VISIBLE -- did you set Runtime > Change runtime type > T4 GPU?"
!apt-get update -qq && apt-get install -y -qq ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-science tipa libcairo2-dev libpango1.0-dev pkg-config python3-dev fonts-thai-tlwg libegl1 libgl1-mesa-glx libgles2-mesa mesa-utils > /dev/null
!pip install -q manim moderngl


In [ ]:
# Headless GPU OpenGL context sanity check BEFORE spending time on real renders.
# On Kaggle this printed "llvmpipe" (CPU software renderer) instead of the real GPU --
# if it prints an NVIDIA name here instead, Colab's container is set up differently.
import moderngl
try:
    ctx = moderngl.create_context(standalone=True, backend='egl')
    print("EGL standalone context OK:", ctx.info.get('GL_RENDERER'))
    ctx.release()
except Exception as e:
    print("EGL standalone context FAILED:", repr(e))
    try:
        ctx = moderngl.create_context(standalone=True)
        print("Default standalone context OK:", ctx.info.get('GL_RENDERER'))
        ctx.release()
    except Exception as e2:
        print("Default standalone context ALSO FAILED:", repr(e2))


In [ ]:
# Public repo, no auth needed (same clone command worked unauthenticated on Kaggle).
!rm -rf /content/manium && git clone --depth 1 https://github.com/minmin2017/manium.git /content/manium
%cd /content/manium


In [ ]:
import os, subprocess, time, json

os.environ['MANIM_THAI_FONT'] = 'Loma'

# Same 3 real, previously-shipped scenes/tiers as the Kaggle run:
#   short  (~10s):  HV12_SolenoidDesign           (hydraulic_valves.py)  -- 9.6s as shipped
#   medium (~53s):  W05_YourMotor_WindAndAssemble (motor_winding.py)     -- 53.2s as shipped
#   long   (~143s): W01..W05 together (motor_winding.py)                 -- the full WIND series, 143.5s as shipped
TIERS = {
    "short_10s": ("hydraulic_valves.py", ["HV12_SolenoidDesign"]),
    "medium_53s": ("motor_winding.py", ["W05_YourMotor_WindAndAssemble"]),
    "long_143s": ("motor_winding.py", [
        "W01_WhyManyCoils_PolePitchVsCommPitch",
        "W02_LapWinding",
        "W03_WaveWinding",
        "W04_Compare_ApplyToYourMotor",
        "W05_YourMotor_WindAndAssemble",
    ]),
}

RENDERERS = {
    "cairo": [],
    "opengl": ["--renderer=opengl", "--write_to_movie"],
}

results = {}
os.makedirs("/content/output", exist_ok=True)

for tier_name, (scene_file, scene_names) in TIERS.items():
    for renderer_name, extra_flags in RENDERERS.items():
        media_dir = f"/tmp/media_{tier_name}_{renderer_name}"
        cmd = [
            "manim", "-qh", "--fps", "30",
            scene_file, *scene_names,
            "--media_dir", media_dir,
            *extra_flags,
        ]
        print("RUNNING:", " ".join(cmd))
        t0 = time.time()
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
        elapsed = time.time() - t0
        key = f"{tier_name}__{renderer_name}"
        results[key] = {
            "elapsed_seconds": round(elapsed, 2),
            "returncode": proc.returncode,
            "stderr_tail": proc.stderr[-2000:],
        }
        print(f"  -> {elapsed:.1f}s, returncode={proc.returncode}")
        if proc.returncode != 0:
            print("  STDERR TAIL:", proc.stderr[-500:])

with open("/content/output/bench_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps({k: {"elapsed_seconds": v["elapsed_seconds"], "returncode": v["returncode"]} for k, v in results.items()}, indent=2))


In [ ]:
import glob, shutil
copied = 0
for tier_name in TIERS:
    for renderer_name in RENDERERS:
        media_dir = f"/tmp/media_{tier_name}_{renderer_name}"
        found = [p for p in glob.glob(f"{media_dir}/**/*.mp4", recursive=True) if "partial_movie_files" not in p]
        for p in found:
            dest = f"/content/output/{tier_name}__{renderer_name}__{os.path.basename(p)}"
            shutil.copy(p, dest)
            copied += 1
            print("copied:", dest)
print("total mp4 copied:", copied)


## Get the results back

Colab sessions are temporary — download the results before closing the tab. Run the cell
below to zip everything in `/content/output/` and pop a browser download prompt. Send the
`bench_results.json` (or just paste its printed content above) back so it can be compared
against the Kaggle numbers.

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/colab_manim_bench_output', 'zip', '/content/output')
files.download('/content/colab_manim_bench_output.zip')
